# EXPT NO: 6
## Implementation of Ensemble Techniques and Handling Class Imbalance

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc, precision_recall_curve

### SCENARIO 1 - Bagging
**Problem Statement:** Predict whether a patient has diabetes using BaggingClassifier.

In [2]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier

# 1. Load dataset
df_diabetes = pd.read_csv('diabetes_bagging.csv')
X_diab = df_diabetes.drop('Outcome', axis=1)
y_diab = df_diabetes['Outcome']
X_train_diab, X_test_diab, y_train_diab, y_test_diab = train_test_split(X_diab, y_diab, test_size=0.2, random_state=42)

# 2. Train Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train_diab, y_train_diab)
dt_pred = dt.predict(X_test_diab)
dt_acc = accuracy_score(y_test_diab, dt_pred)

# 3. Apply BaggingClassifier
bagging = BaggingClassifier(base_estimator=dt, n_estimators=50, random_state=42)
bagging.fit(X_train_diab, y_train_diab)
bagging_pred = bagging.predict(X_test_diab)
bagging_acc = accuracy_score(y_test_diab, bagging_pred)

print(f"Decision Tree Accuracy: {dt_acc:.4f}")
print(f"Bagging Accuracy: {bagging_acc:.4f}")

# 4. Visualization
plt.figure(figsize=(12, 5))

# Accuracy comparison bar graph
plt.subplot(1, 2, 1)
plt.bar(['Decision Tree', 'Bagging'], [dt_acc, bagging_acc], color=['skyblue', 'lightgreen'])
plt.title('Accuracy Comparison')
plt.ylabel('Accuracy')

# Confusion matrix for Bagging
plt.subplot(1, 2, 2)
sns.heatmap(confusion_matrix(y_test_diab, bagging_pred), annot=True, fmt='d', cmap='Blues')
plt.title('Bagging Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')

plt.tight_layout()
plt.show()

TypeError: BaggingClassifier.__init__() got an unexpected keyword argument 'base_estimator'

### SCENARIO 2 - Boosting (AdaBoost, Gradient Boosting)
**Problem Statement:** Predict whether a customer will churn.

In [ ]:
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder

# 1. Load dataset
df_churn = pd.read_csv('churn_boosting.csv')

# Very basic data prep if there are categorical features (handle appropriately based on your actual data)
df_churn.fillna(df_churn.mode().iloc[0], inplace=True)
for col in df_churn.select_dtypes(include='object').columns:
    df_churn[col] = LabelEncoder().fit_transform(df_churn[col])

target_col = 'Churn' if 'Churn' in df_churn.columns else df_churn.columns[-1]
X_churn = df_churn.drop(target_col, axis=1)
y_churn = df_churn[target_col]
X_train_ch, X_test_ch, y_train_ch, y_test_ch = train_test_split(X_churn, y_churn, test_size=0.2, random_state=42)

# 2. Train AdaBoost & Gradient Boosting
ada = AdaBoostClassifier(random_state=42)
ada.fit(X_train_ch, y_train_ch)
ada_pred = ada.predict(X_test_ch)
ada_probs = ada.predict_proba(X_test_ch)[:, 1]

gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train_ch, y_train_ch)
gb_pred = gb.predict(X_test_ch)
gb_probs = gb.predict_proba(X_test_ch)[:, 1]

print(f"AdaBoost Accuracy: {accuracy_score(y_test_ch, ada_pred):.4f}")
print(f"Gradient Boosting Accuracy: {accuracy_score(y_test_ch, gb_pred):.4f}")

# 3. Visualization: ROC Curve & Feature Importance plot
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
fpr_ada, tpr_ada, _ = roc_curve(y_test_ch, ada_probs)
fpr_gb, tpr_gb, _ = roc_curve(y_test_ch, gb_probs)
plt.plot(fpr_ada, tpr_ada, label=f'AdaBoost (AUC = {auc(fpr_ada, tpr_ada):.2f})')
plt.plot(fpr_gb, tpr_gb, label=f'GradBoost (AUC = {auc(fpr_gb, tpr_gb):.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.title('ROC Curve')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()

plt.subplot(1, 2, 2)
feat_imp = pd.Series(gb.feature_importances_, index=X_churn.columns).sort_values(ascending=False).head(10)
feat_imp.plot(kind='bar', color='coral')
plt.title('Gradient Boosting - Feature Importance')

plt.tight_layout()
plt.show()

### SCENARIO 3 - Random Forest
**Problem Statement:** Predict whether a person earns >50K salary.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 1. Load dataset
df_income = pd.read_csv('income_random_forest.csv')
# Encoding basics
for col in df_income.select_dtypes(include='object').columns:
    df_income[col] = LabelEncoder().fit_transform(df_income[col])

target_col = 'Income' if 'Income' in df_income.columns else df_income.columns[-1]
X_inc = df_income.drop(target_col, axis=1)
y_inc = df_income[target_col]
X_train_inc, X_test_inc, y_train_inc, y_test_inc = train_test_split(X_inc, y_inc, test_size=0.2, random_state=42)

# 2. Train Random Forest & Tune number of trees
n_estimators_list = [10, 50, 100, 200]
accuracies = []

for n in n_estimators_list:
    rf = RandomForestClassifier(n_estimators=n, random_state=42)
    rf.fit(X_train_inc, y_train_inc)
    preds = rf.predict(X_test_inc)
    accuracies.append(accuracy_score(y_test_inc, preds))
    
# Final evaluation on best model (100 in this case for demo)
best_rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train_inc, y_train_inc)

# 3. Visualization
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
feat_imp_rf = pd.Series(best_rf.feature_importances_, index=X_inc.columns).sort_values(ascending=False).head(10)
feat_imp_rf.plot(kind='bar', color='teal')
plt.title('Random Forest - Feature Importance')

plt.subplot(1, 2, 2)
plt.plot(n_estimators_list, accuracies, marker='o', linestyle='-', color='purple')
plt.title('Accuracy vs Number of Trees')
plt.xlabel('Number of Trees')
plt.ylabel('Accuracy')

plt.tight_layout()
plt.show()

### SCENARIO 4 - Stacking
**Problem Statement:** Predict heart disease presence.

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# 1. Load dataset
df_heart = pd.read_csv('heart_stacking.csv')
target_col = 'target' if 'target' in df_heart.columns else df_heart.columns[-1]
X_ht = df_heart.drop(target_col, axis=1)
y_ht = df_heart[target_col]
X_train_ht, X_test_ht, y_train_ht, y_test_ht = train_test_split(X_ht, y_ht, test_size=0.2, random_state=42)

# 2. Train base models 
estimators = [
    ('lr', LogisticRegression(max_iter=1000)),
    ('svm', SVC(probability=True)),
    ('dt', DecisionTreeClassifier())
]

model_accs = {}
for name, model in estimators:
    model.fit(X_train_ht, y_train_ht)
    model_accs[name] = accuracy_score(y_test_ht, model.predict(X_test_ht))

# 3. Combine using StackingClassifier
stack_model = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression())
stack_model.fit(X_train_ht, y_train_ht)
stack_acc = accuracy_score(y_test_ht, stack_model.predict(X_test_ht))
model_accs['Stacking'] = stack_acc

for name, acc in model_accs.items():
    print(f"{name} Accuracy: {acc:.4f}")

# 4. Visualization: Model comparison bar chart
plt.figure(figsize=(8, 5))
plt.bar(model_accs.keys(), model_accs.values(), color=['salmon', 'lightblue', 'lightgreen', 'purple'])
plt.title('Base Models vs Stacking Classification')
plt.ylabel('Accuracy')
plt.show()

### SCENARIO 5 - SMOTE
**Problem Statement:** Detect fraudulent transactions handling class imbalance.

In [ ]:
# Install imblearn if not already installed -> !pip install imbalanced-learn
from imblearn.over_sampling import SMOTE

# 1. Load dataset
df_fraud = pd.read_csv('fraud_smote.csv')
target_col = 'Class' if 'Class' in df_fraud.columns else df_fraud.columns[-1]
X_fr = df_fraud.drop(target_col, axis=1)
y_fr = df_fraud[target_col]

X_train_fr, X_test_fr, y_train_fr, y_test_fr = train_test_split(X_fr, y_fr, test_size=0.2, random_state=42)

# 2. Train model BEFORE SMOTE
lr_before = LogisticRegression(max_iter=1000)
lr_before.fit(X_train_fr, y_train_fr)
pred_before = lr_before.predict(X_test_fr)
probs_before = lr_before.predict_proba(X_test_fr)[:, 1]

# 3. Apply SMOTE
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_fr, y_train_fr)

# Train model AFTER SMOTE
lr_after = LogisticRegression(max_iter=1000)
lr_after.fit(X_train_sm, y_train_sm)
pred_after = lr_after.predict(X_test_fr)
probs_after = lr_after.predict_proba(X_test_fr)[:, 1]

# 4. Visualization
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
dist_before = y_train_fr.value_counts()
dist_after = pd.Series(y_train_sm).value_counts()
pd.DataFrame({'Before SMOTE': dist_before, 'After SMOTE': dist_after}).plot(kind='bar', ax=plt.gca())
plt.title('Class Distribution (Before & After SMOTE)')

plt.subplot(1, 2, 2)
prec_bef, rec_bef, _ = precision_recall_curve(y_test_fr, probs_before)
prec_aft, rec_aft, _ = precision_recall_curve(y_test_fr, probs_after)
plt.plot(rec_bef, prec_bef, label='Before SMOTE')
plt.plot(rec_aft, prec_aft, label='After SMOTE')
plt.title('Precision-Recall Curve')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend()

plt.tight_layout()
plt.show()

print("\n--- BEFORE SMOTE ---")
print(classification_report(y_test_fr, pred_before))
print("\n--- AFTER SMOTE ---")
print(classification_report(y_test_fr, pred_after))